# Gold de atendimentos — V-Energy

Publica os atendimentos das UCs no formato do contrato do portal (`contracts/tabelas/`), a partir da Silver de atendimentos. A transformação inteira está neste notebook (sem `%run` e sem módulo `.py` externo). Produz as duas tabelas do contrato:

1. `atendimentos`: um protocolo por linha, com a UC, a abertura, o assunto, a situação e a nota de satisfação (de 1 a 5, uma casa decimal; nula quando o titular não avaliou).
2. `atendimentos_historico`: uma etapa por linha (protocolo × sequência), com a data e a ação registrada, para a linha do tempo de cada atendimento.

Regras e dicionário de dados: [`docs/gold/atendimentos.md`](../../docs/gold/atendimentos.md).

O padrão `modo=validar` não grava nada: **Run all** valida as fontes, monta e confere as duas tabelas em memória. Para gravar em `v-energy.gold`, o Job passa `modo=publicar` (ou troque o widget antes de executar), como nas outras Golds.

In [0]:
"""Transformações e validações da Gold de atendimentos."""

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType, IntegerType

PADRAO_ID_UC = r'^UC\d{6}\z'
STATUS_ENCERRADO = 'ENCERRADO'
STATUS_EM_ABERTO = 'EM ABERTO'
# Escala do contrato (atendimentos.nota, numeric(2,1)): de 1 a 5, uma casa decimal
NOTA_CONTRATO = DecimalType(2, 1)
NOTA_MINIMA, NOTA_MAXIMA = 1, 5
FORMATO_DATA_HORA = 'yyyy-MM-dd HH:mm:ss'
COLUNAS_ATENDIMENTOS = [
    'protocolo',
    'id_uc',
    'assunto',
    'data_abertura',
    'data_fechamento',
    'nota_satisfacao',
    'historico',
]
# Colunas, tipos e ordem de contracts/tabelas/<tabela>.yaml
COLUNAS_CONTRATO = {
    'atendimentos': [
        ('protocolo', 'string'),
        ('id_uc', 'string'),
        ('aberto_em', 'timestamp_ntz'),
        ('assunto', 'string'),
        ('status', 'string'),
        ('nota', 'decimal(2,1)'),
    ],
    'atendimentos_historico': [
        ('protocolo', 'string'),
        ('sequencia', 'int'),
        ('data', 'timestamp_ntz'),
        ('descricao', 'string'),
    ],
}


def contar(df: DataFrame, regras: dict) -> dict:
    """Conta, numa única passada, as linhas que violam cada regra."""
    linha = df.agg(*[F.count(F.when(cond, 1)).alias(nome) for nome, cond in regras.items()])
    return linha.first().asDict()


def checar(violacoes: dict, rotulo: str) -> None:
    """Imprime cada regra e falha se alguma tiver violação."""
    for regra, n in violacoes.items():
        print(f'[{"FALHA" if n else "OK   "}] ({rotulo}) {regra}: {n:,}')
    falhas = {regra: n for regra, n in violacoes.items() if n}
    if falhas:
        raise RuntimeError(f'Validações bloqueantes falharam ({rotulo}): {falhas}')


def fora_do_padrao(coluna: str, padrao: str):
    return F.col(coluna).isNull() | ~F.col(coluna).rlike(padrao)


def vazio(coluna: str):
    return F.coalesce(F.trim(F.col(coluna)), F.lit('')) == ''


def nota_invalida(coluna: str):
    """Nota preenchida fora de 1 a 5 ou com mais de uma casa decimal; nula é válida."""
    nota = F.col(coluna)
    return nota.isNotNull() & (
        (nota < NOTA_MINIMA) | (nota > NOTA_MAXIMA) | (F.round(nota, 1) != nota)
    )


def eventos_da_silver(atendimentos: DataFrame) -> DataFrame:
    return atendimentos.select('protocolo', F.explode('historico').alias('evento')).select(
        'protocolo',
        F.col('evento.sequencia').alias('sequencia'),
        F.col('evento.data').alias('data'),
        F.col('evento.acao').alias('acao'),
    )


def validar_fontes(atendimentos: DataFrame, cadastro: DataFrame) -> None:
    """Colunas, chaves, domínios e cruzamento com o cadastro, antes de qualquer escrita."""
    ausentes = sorted(set(COLUNAS_ATENDIMENTOS) - set(atendimentos.columns))
    if ausentes:
        raise RuntimeError(f'Silver de atendimentos sem as colunas {ausentes}')
    if 'id_uc' not in cadastro.columns:
        raise RuntimeError('Silver de cadastro sem a coluna id_uc')

    eventos = eventos_da_silver(atendimentos)
    violacoes = contar(
        atendimentos,
        {
            'atendimentos: protocolo nulo ou vazio': vazio('protocolo'),
            'atendimentos: id_uc fora do padrão UC + 6 dígitos': fora_do_padrao(
                'id_uc', PADRAO_ID_UC
            ),
            'atendimentos: assunto nulo ou vazio': vazio('assunto'),
            'atendimentos: data de abertura nula': F.col('data_abertura').isNull(),
            'atendimentos: fechamento antes da abertura': F.col('data_fechamento')
            < F.col('data_abertura'),
            'atendimentos: nota fora de 1 a 5 ou com mais de uma casa decimal': nota_invalida(
                'nota_satisfacao'
            ),
        },
    )
    violacoes |= contar(
        eventos,
        {
            'histórico: sequência nula ou menor que 1': F.col('sequencia').isNull()
            | (F.col('sequencia') < 1),
            'histórico: data nula': F.col('data').isNull(),
            'histórico: ação nula ou vazia': vazio('acao'),
        },
    )
    violacoes |= {
        'atendimentos: protocolo repetido (linhas a mais)': atendimentos.count()
        - atendimentos.select('protocolo').distinct().count(),
        'histórico: protocolo × sequência repetido (linhas a mais)': eventos.count()
        - eventos.select('protocolo', 'sequencia').distinct().count(),
        'atendimentos: id_uc ausente da Silver de cadastro': atendimentos.join(
            cadastro.select('id_uc'), 'id_uc', 'left_anti'
        ).count(),
    }
    checar(violacoes, 'fontes')


def construir_gold_atendimentos(atendimentos: DataFrame) -> dict:
    """Monta as duas tabelas do contrato a partir da Silver de atendimentos."""
    # A Silver guarda a data e hora do arquivo de origem como TIMESTAMP; a Gold usa
    # TIMESTAMP_NTZ, como as outras Golds, para a hora não depender de fuso na exportação.
    gold_atendimentos = atendimentos.select(
        'protocolo',
        'id_uc',
        F.col('data_abertura').cast('timestamp_ntz').alias('aberto_em'),
        'assunto',
        F.when(F.col('data_fechamento').isNull(), F.lit(STATUS_EM_ABERTO))
        .otherwise(F.lit(STATUS_ENCERRADO))
        .alias('status'),
        F.col('nota_satisfacao').cast(NOTA_CONTRATO).alias('nota'),
    )
    gold_historico = eventos_da_silver(atendimentos).select(
        'protocolo',
        F.col('sequencia').cast(IntegerType()).alias('sequencia'),
        F.col('data').cast('timestamp_ntz').alias('data'),
        F.col('acao').alias('descricao'),
    )
    return {'atendimentos': gold_atendimentos, 'atendimentos_historico': gold_historico}


def validar_gold(tabelas: dict, atendimentos: DataFrame, cadastro: DataFrame, rotulo: str) -> None:
    """Contrato, chaves e reconciliação das duas tabelas contra a Silver."""
    gold_atendimentos = tabelas['atendimentos']
    gold_historico = tabelas['atendimentos_historico']
    for nome, df in tabelas.items():
        colunas = [(c.name, c.dataType.simpleString()) for c in df.schema]
        if colunas != COLUNAS_CONTRATO[nome]:
            raise RuntimeError(f'{nome}: colunas, tipos ou ordem fora do contrato: {colunas}')

    tamanho_historico = F.when(F.col('historico').isNull(), 0).otherwise(F.size('historico'))
    fonte = atendimentos.agg(
        F.count(F.lit(1)).alias('linhas'),
        F.count(F.when(F.col('nota_satisfacao').isNull(), 1)).alias('sem_nota'),
        F.count(F.when(F.col('data_fechamento').isNull(), 1)).alias('em_aberto'),
        F.sum(tamanho_historico).alias('eventos'),
    ).first()

    obrigatorias = [c for c, _ in COLUNAS_CONTRATO['atendimentos'] if c != 'nota']
    violacoes = contar(
        gold_atendimentos,
        {
            'atendimentos: coluna obrigatória nula': F.greatest(
                *[F.col(c).isNull() for c in obrigatorias]
            ),
            'atendimentos: id_uc fora do padrão': fora_do_padrao('id_uc', PADRAO_ID_UC),
            'atendimentos: protocolo ou assunto vazio': vazio('protocolo') | vazio('assunto'),
            'atendimentos: status fora do domínio': ~F.col('status').isin(
                STATUS_ENCERRADO, STATUS_EM_ABERTO
            ),
            'atendimentos: nota fora de 1 a 5': nota_invalida('nota'),
        },
    )
    violacoes |= contar(
        gold_historico,
        {
            'atendimentos_historico: coluna obrigatória nula': F.greatest(
                *[F.col(c).isNull() for c in gold_historico.columns]
            ),
            'atendimentos_historico: sequência menor que 1': F.col('sequencia') < 1,
            'atendimentos_historico: descrição vazia': vazio('descricao'),
        },
    )

    total = gold_atendimentos.agg(
        F.count(F.lit(1)).alias('linhas'),
        F.countDistinct('protocolo').alias('ids'),
        F.count(F.when(F.col('nota').isNull(), 1)).alias('sem_nota'),
        F.count(F.when(F.col('status') == STATUS_EM_ABERTO, 1)).alias('em_aberto'),
    ).first()
    total_historico = gold_historico.agg(
        F.count(F.lit(1)).alias('linhas'),
        F.countDistinct('protocolo', 'sequencia').alias('chaves'),
    ).first()
    silver = atendimentos.select(
        'protocolo',
        'nota_satisfacao',
        F.date_format('data_abertura', FORMATO_DATA_HORA).alias('abertura_silver'),
    )
    comparado = gold_atendimentos.join(silver, 'protocolo', 'left')
    nota_diferente = ~F.col('nota').isNull().eqNullSafe(F.col('nota_satisfacao').isNull()) | (
        F.abs(F.col('nota').cast('double') - F.col('nota_satisfacao')) > 1e-9
    )
    violacoes |= {
        'atendimentos: linhas diferentes da Silver': abs(total['linhas'] - fonte['linhas']),
        'atendimentos: protocolo repetido (linhas a mais)': total['linhas'] - total['ids'],
        'atendimentos: protocolo ausente da Silver': comparado.filter(
            F.col('abertura_silver').isNull()
        ).count(),
        'atendimentos: nota diferente da Silver': comparado.filter(nota_diferente).count(),
        'atendimentos: aberto_em com data ou hora diferente da Silver': comparado.filter(
            F.date_format('aberto_em', FORMATO_DATA_HORA) != F.col('abertura_silver')
        ).count(),
        'atendimentos: em aberto diferente dos protocolos sem fechamento': abs(
            total['em_aberto'] - fonte['em_aberto']
        ),
        'atendimentos: id_uc ausente do cadastro (dim_uc)': gold_atendimentos.join(
            cadastro.select('id_uc'), 'id_uc', 'left_anti'
        ).count(),
        'atendimentos_historico: linhas diferentes dos eventos da Silver': abs(
            total_historico['linhas'] - fonte['eventos']
        ),
        'atendimentos_historico: protocolo × sequência repetido (linhas a mais)': total_historico[
            'linhas'
        ]
        - total_historico['chaves'],
        'atendimentos_historico: protocolo ausente de atendimentos': gold_historico.join(
            gold_atendimentos.select('protocolo'), 'protocolo', 'left_anti'
        ).count(),
    }
    checar(violacoes, rotulo)
    print(
        f'Reconciliação ({rotulo}): {fonte["linhas"]:,} atendimentos | '
        f'{fonte["eventos"]:,} etapas no histórico | {fonte["sem_nota"]:,} sem nota | '
        f'{fonte["em_aberto"]:,} em aberto'
    )


def publicar_delta(spark, tabelas: dict, destinos: dict, validar) -> tuple:
    """Publica e confere as tabelas Gold; recupera os destinos afetados em caso de falha."""
    anteriores = {}
    for nome in tabelas:
        destino = destinos[nome]
        if spark.catalog.tableExists(destino):
            detalhe = spark.sql(f'DESCRIBE DETAIL {destino}').first()
            if detalhe['format'] != 'delta':
                raise RuntimeError(f'O destino precisa ser Delta: {destino}')
            anteriores[nome] = spark.sql(f'DESCRIBE HISTORY {destino} LIMIT 1').first()['version']
        else:
            anteriores[nome] = None

    afetadas = []
    persistidas, versoes = {}, {}
    try:
        for nome, df in tabelas.items():
            destino = destinos[nome]
            afetadas.append(nome)
            (
                df.write.format('delta')
                .mode('overwrite')
                .option('overwriteSchema', 'true')
                .saveAsTable(destino)
            )
            versao = spark.sql(f'DESCRIBE HISTORY {destino} LIMIT 1').first()['version']
            persistido = spark.read.option('versionAsOf', versao).table(destino)
            esperado = [(c.name, c.dataType.simpleString()) for c in df.schema]
            gravado = [(c.name, c.dataType.simpleString()) for c in persistido.schema]
            if esperado != gravado:
                raise RuntimeError(f'Schema persistido divergente: {destino}')
            if (
                df.exceptAll(persistido).limit(1).count()
                or persistido.exceptAll(df).limit(1).count()
            ):
                raise RuntimeError(f'Conteúdo persistido divergente: {destino}')
            persistidas[nome] = persistido
            versoes[nome] = {'tabela': destino, 'versao_delta': versao}
        validar(persistidas)
    except Exception as erro:
        falhas = []
        for nome in reversed(afetadas):
            destino = destinos[nome]
            try:
                versao = anteriores[nome]
                if versao is None:
                    spark.sql(f'DROP TABLE IF EXISTS {destino}')
                else:
                    spark.sql(f'RESTORE TABLE {destino} TO VERSION AS OF {versao}')
            except Exception as falha:
                falhas.append(RuntimeError(f'Falha ao recuperar {destino}: {falha}'))
        if falhas:
            raise ExceptionGroup(
                'Publicação falhou e a recuperação ficou incompleta', [erro, *falhas]
            ) from erro
        raise
    return persistidas, versoes

In [0]:
# ruff: noqa: F821
import re

parametros = {
    'tabela_atendimentos': 'v-energy.silver.tb_atendimentos',
    'tabela_cadastro': 'v-energy.silver.tb_unidades_consumidoras',
    'schema_gold': 'v-energy.gold',
    'modo': 'validar',
}
for nome, padrao in parametros.items():
    dbutils.widgets.text(nome, padrao)
config = {nome: dbutils.widgets.get(nome).strip() for nome in parametros}
if config['modo'] not in {'validar', 'publicar'}:
    raise RuntimeError('modo deve ser validar ou publicar')


def qualificar(nome, partes=3):
    segmentos = nome.split('.')
    if len(segmentos) != partes or any(
        not re.fullmatch(r'[A-Za-z0-9_-]+', segmento) for segmento in segmentos
    ):
        raise RuntimeError(f'Identificador inválido: {nome}')
    return '.'.join(f'`{segmento}`' for segmento in segmentos)


schema_gold = qualificar(config['schema_gold'], 2)
if config['schema_gold'].split('.')[1] in {'silver', 'bronze', 'default'}:
    raise RuntimeError('O destino deve ser um schema próprio da Gold')
fontes = {}
print(f'Schema Gold: {schema_gold} | Modo: {config["modo"]}')
if config['modo'] == 'publicar':
    print(f'ATENÇÃO: modo=publicar. As tabelas Gold serão gravadas em {schema_gold}')

Schema Gold: `v-energy`.`gold` | Modo: publicar
ATENÇÃO: modo=publicar. As tabelas Gold serão gravadas em `v-energy`.`gold`


## 1. Leitura consistente da Silver (`VERSION AS OF`) e validação

Fixa a versão Delta atual das duas Silvers no início da execução, para que validações, transformações e escrita leiam exatamente os mesmos dados. Antes de montar a Gold, confere colunas, chaves e domínios da Silver de atendimentos e o cruzamento com o cadastro: toda UC atendida existe na Silver de cadastro, de onde sai a `dim_uc`. Depois de montar, confere o contrato das duas tabelas e a reconciliação com a Silver.

In [0]:
# ruff: noqa: F821
def ler_snapshot(chave):
    tabela = qualificar(config[chave])
    if not spark.catalog.tableExists(tabela):
        raise RuntimeError(
            f'Tabela Silver não encontrada: {tabela}. '
            f'Publique a Silver correspondente ou ajuste o widget {chave}.'
        )
    detalhe = spark.sql(f'DESCRIBE DETAIL {tabela}').first()
    if detalhe['format'] != 'delta':
        raise RuntimeError(f'A fonte precisa ser Delta: {tabela}')
    historico = spark.sql(f'DESCRIBE HISTORY {tabela} LIMIT 1').first()
    fontes[chave] = {
        'tabela': config[chave],
        'id_delta': detalhe['id'],
        'versao_delta': historico['version'],
    }
    print(f'{chave}: {tabela} (versão {historico["version"]})')
    return spark.read.option('versionAsOf', historico['version']).table(tabela)


df_atendimentos = ler_snapshot('tabela_atendimentos')
df_cadastro = ler_snapshot('tabela_cadastro')

validar_fontes(df_atendimentos, df_cadastro)
tabelas = construir_gold_atendimentos(df_atendimentos)
validar_gold(tabelas, df_atendimentos, df_cadastro, 'antes de gravar')
for nome, df_tab in tabelas.items():
    print(f'[OK] {nome}: {df_tab.count():,} registros validados em memória')

tabela_atendimentos: `v-energy`.`silver`.`tb_atendimentos` (versão 0)
tabela_cadastro: `v-energy`.`silver`.`tb_unidades_consumidoras` (versão 43)
[OK   ] (fontes) atendimentos: protocolo nulo ou vazio: 0
[OK   ] (fontes) atendimentos: id_uc fora do padrão UC + 6 dígitos: 0
[OK   ] (fontes) atendimentos: assunto nulo ou vazio: 0
[OK   ] (fontes) atendimentos: data de abertura nula: 0
[OK   ] (fontes) atendimentos: fechamento antes da abertura: 0
[OK   ] (fontes) atendimentos: nota fora de 1 a 5 ou com mais de uma casa decimal: 0
[OK   ] (fontes) histórico: sequência nula ou menor que 1: 0
[OK   ] (fontes) histórico: data nula: 0
[OK   ] (fontes) histórico: ação nula ou vazia: 0
[OK   ] (fontes) atendimentos: protocolo repetido (linhas a mais): 0
[OK   ] (fontes) histórico: protocolo × sequência repetido (linhas a mais): 0
[OK   ] (fontes) atendimentos: id_uc ausente da Silver de cadastro: 0
[OK   ] (antes de gravar) atendimentos: coluna obrigatória nula: 0
[OK   ] (antes de gravar) aten

## 2. Verificação da pergunta da Gerente de Atendimento

**Camila Barreto:** _"Tem que aparecer tudo numa tela só: quem é o cliente, (...) e os chamados que ele já abriu."_ Na Visão 360, o bloco de chamados mostra os protocolos, o assunto, a nota e a linha do tempo das tratativas. Confere nas tabelas geradas:
- volume, nota média e parcela sem nota por assunto;
- a nota média por ano de abertura;
- o bloco de chamados de uma UC, montado só com as duas tabelas;
- as etapas registradas no histórico.

In [0]:
# ruff: noqa: F821
gold_atendimentos = tabelas['atendimentos']
gold_historico = tabelas['atendimentos_historico']
sem_nota = F.col('nota').isNull().cast('int')

print('=== 1) Atendimentos por assunto: volume, nota média e parcela sem nota ===')
display(
    gold_atendimentos.groupBy('assunto')
    .agg(
        F.count(F.lit(1)).alias('atendimentos'),
        F.round(F.avg('nota'), 2).alias('nota_media'),
        F.round(100 * F.avg(sem_nota), 1).alias('pct_sem_nota'),
    )
    .orderBy(F.desc('atendimentos'), 'assunto')
)

print('=== 2) Nota média por ano de abertura ===')
display(
    gold_atendimentos.groupBy(F.year('aberto_em').alias('ano'))
    .agg(
        F.count(F.lit(1)).alias('atendimentos'),
        F.round(F.avg('nota'), 2).alias('nota_media'),
        F.round(100 * F.avg(sem_nota), 1).alias('pct_sem_nota'),
    )
    .orderBy('ano')
)

print('=== 3) Bloco de chamados da Visão 360: a UC com mais protocolos ===')
uc_exemplo = (
    gold_atendimentos.groupBy('id_uc').count().orderBy(F.desc('count'), 'id_uc').first()['id_uc']
)
print(f'UC: {uc_exemplo}')
display(
    gold_atendimentos.filter(F.col('id_uc') == uc_exemplo)
    .join(gold_historico, 'protocolo')
    .select('protocolo', 'aberto_em', 'assunto', 'status', 'nota', 'sequencia', 'data', 'descricao')
    .orderBy('aberto_em', 'protocolo', 'sequencia')
)

print('=== 4) Etapas registradas no histórico ===')
display(
    gold_historico.groupBy('descricao')
    .agg(F.count(F.lit(1)).alias('etapas'))
    .orderBy(F.desc('etapas'), 'descricao')
)

=== 1) Atendimentos por assunto: volume, nota média e parcela sem nota ===


assunto,atendimentos,nota_media,pct_sem_nota
FALTA DE ENERGIA,101306,4.04,3.0
FATURA,43624,4.05,3.0
DANO ELETRICO,43582,4.04,3.0
LIGACAO NOVA,43520,4.04,3.0
GERACAO DISTRIBUIDA,43499,4.05,2.9
RELIGACAO,43340,4.04,3.0


=== 2) Nota média por ano de abertura ===


ano,atendimentos,nota_media,pct_sem_nota
2022,38275,4.10,3.1
2023,76928,4.10,2.9
2024,76750,4.10,3.1
2025,88212,3.90,3.0
2026,38706,4.10,2.8


=== 3) Bloco de chamados da Visão 360: a UC com mais protocolos ===
UC: UC131266


protocolo,aberto_em,assunto,status,nota,sequencia,data,descricao
AT000004981,2022-07-14T12:13:00.000,FALTA DE ENERGIA,ENCERRADO,3.9,1,2022-07-14T22:13:00.000,ENCERRAMENTO
AT000004981,2022-07-14T12:13:00.000,FALTA DE ENERGIA,ENCERRADO,3.9,2,2022-07-15T08:13:00.000,RETORNO AO CLIENTE
AT000004981,2022-07-14T12:13:00.000,FALTA DE ENERGIA,ENCERRADO,3.9,3,2022-07-15T18:13:00.000,DESPACHO
AT000006075,2022-07-31T02:22:00.000,LIGACAO NOVA,ENCERRADO,3.9,1,2022-08-03T02:22:00.000,ANALISE
AT000016982,2022-09-07T11:10:00.000,FATURA,ENCERRADO,4.3,1,2022-09-10T11:10:00.000,DESPACHO
AT000016982,2022-09-07T11:10:00.000,FATURA,ENCERRADO,4.3,2,2022-09-13T11:10:00.000,DESPACHO
AT000016982,2022-09-07T11:10:00.000,FATURA,ENCERRADO,4.3,3,2022-09-16T12:10:00.000,ANALISE
AT000028929,2022-11-20T17:42:00.000,GERACAO DISTRIBUIDA,ENCERRADO,4.3,1,2022-11-25T14:42:00.000,ENCERRAMENTO
AT000028929,2022-11-20T17:42:00.000,GERACAO DISTRIBUIDA,ENCERRADO,4.3,2,2022-11-30T11:42:00.000,ENCERRAMENTO
AT000049408,2023-02-27T06:03:00.000,FATURA,ENCERRADO,3.6,1,2023-03-02T16:03:00.000,RETORNO AO CLIENTE


=== 4) Etapas registradas no histórico ===


descricao,etapas
ABERTURA,127796
RETORNO AO CLIENTE,127484
DESPACHO,127417
ENCERRAMENTO,127256
ANALISE,126904


## 3. Publicação Delta com `RESTORE`

A publicação só ocorre quando `modo=publicar`. Antes da primeira escrita, registra a versão atual de cada tabela de destino. Depois de gravar, confere o schema e o conteúdo gravados e repete a validação da seção 1 sobre as tabelas persistidas; se a escrita ou a conferência falhar, restaura os destinos afetados com `RESTORE TABLE ... TO VERSION AS OF` (ou `DROP TABLE IF EXISTS` na primeira carga).

In [0]:
# ruff: noqa: F821
if config['modo'] == 'validar':
    print('Validação concluída. Nenhum schema ou tabela foi gravado.')
else:
    if any(dbutils.widgets.get(nome).strip() != valor for nome, valor in config.items()):
        raise RuntimeError(
            'Parâmetros mudaram durante a execução. Execute novamente desde o início.'
        )
    destinos = {nome: f'{schema_gold}.`{nome}`' for nome in tabelas}
    for chave, fonte in fontes.items():
        atual = spark.sql(f'DESCRIBE DETAIL {qualificar(config[chave])}').first()
        if atual['id'] != fonte['id_delta']:
            raise RuntimeError(f'A identidade da tabela fonte mudou: {chave}')
    if set(destinos.values()) & {qualificar(config[c]) for c in fontes}:
        raise RuntimeError('Uma tabela de destino coincide com uma fonte')

    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {schema_gold}')
    persistidas, versoes_gold = publicar_delta(
        spark,
        tabelas,
        destinos,
        lambda gravadas: validar_gold(gravadas, df_atendimentos, df_cadastro, 'tabela gravada'),
    )
    for nome, info in versoes_gold.items():
        print(f'Publicada: {info["tabela"]} (versão {info["versao_delta"]})')

[OK   ] (tabela gravada) atendimentos: coluna obrigatória nula: 0
[OK   ] (tabela gravada) atendimentos: id_uc fora do padrão: 0
[OK   ] (tabela gravada) atendimentos: protocolo ou assunto vazio: 0
[OK   ] (tabela gravada) atendimentos: status fora do domínio: 0
[OK   ] (tabela gravada) atendimentos: nota fora de 1 a 5: 0
[OK   ] (tabela gravada) atendimentos_historico: coluna obrigatória nula: 0
[OK   ] (tabela gravada) atendimentos_historico: sequência menor que 1: 0
[OK   ] (tabela gravada) atendimentos_historico: descrição vazia: 0
[OK   ] (tabela gravada) atendimentos: linhas diferentes da Silver: 0
[OK   ] (tabela gravada) atendimentos: protocolo repetido (linhas a mais): 0
[OK   ] (tabela gravada) atendimentos: protocolo ausente da Silver: 0
[OK   ] (tabela gravada) atendimentos: nota diferente da Silver: 0
[OK   ] (tabela gravada) atendimentos: aberto_em com data ou hora diferente da Silver: 0
[OK   ] (tabela gravada) atendimentos: em aberto diferente dos protocolos sem fechame